In [ ]:
# data-5.csv holds 400 flagged transactions of the exchange, one per row, with 16 features that the exchange's 
# app computes before a transfer is released, and the target fraud (1 = later confirmed fraud, 0 = clean)

In [ ]:
import pandas as pd

#Modify the path according to where the file is in your filesystem
dataset_path = "data-5.csv" 

dataset = pd.read_csv(dataset_path)

In [ ]:
import pandasql as ps

# function query(sql) will help us write our SQL queries easily
def query(sql):
    return ps.sqldf(sql, globals())

In [ ]:
# Column types listed
dataset.dtypes

# 15 number columns and 1 text column (kyc_tier)

amount                       float64
tx_hour                        int64
input_count                    int64
output_count                   int64
wallet_age_days                int64
tx_count_24h                   int64
unique_counterparties_24h      int64
avg_inter_tx_interval_s        int64
std_inter_tx_interval_s        int64
balance_change               float64
new_address_fraction         float64
max_graph_hop                  int64
time_since_prev_tx_s           int64
gas_price_pctile               int64
is_weekend                     int64
kyc_tier                         str
fraud                          int64
dtype: object

In [ ]:
 # checking for missing values

cols_with_nan = dataset.columns[dataset.isna().any()].tolist()
print (cols_with_nan)  

#[] this means that there are no NaN values in the whole dataframe

[]


In [ ]:
# here we are checking whether there are outliers and how many 

num_cols = dataset.select_dtypes("number").columns.drop("fraud")

for col in num_cols:

    #formula for the outliers 
    q1, q3 = dataset[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        
    n_out = 0 # check the number of outliers per column 
    
    for value in dataset[col]:
        if value < lower:
            n_out = n_out + 1
        elif value > upper:
            n_out = n_out + 1
            
    
    print(col,"outliers:", n_out)

amount outliers: 49
tx_hour outliers: 0
input_count outliers: 0
output_count outliers: 0
wallet_age_days outliers: 42
tx_count_24h outliers: 28
unique_counterparties_24h outliers: 0
avg_inter_tx_interval_s outliers: 39
std_inter_tx_interval_s outliers: 38
balance_change outliers: 103
new_address_fraction outliers: 3
max_graph_hop outliers: 0
time_since_prev_tx_s outliers: 46
gas_price_pctile outliers: 0
is_weekend outliers: 0


In [ ]:
# amount: a transaction amount can't be negative
display(query("""
SELECT amount 
FROM dataset
WHERE amount < 0;
"""))         # we dont have an incosistency in amount 

# tx_hour: hours only go from 0 to 23
display(query("""
SELECT tx_hour
FROM dataset
WHERE tx_hour < 0 OR tx_hour > 23;
"""))

# input_count: a transaction needs at least 1 input
display(query("""
SELECT input_count
FROM dataset
WHERE input_count < 1;
"""))

# output_count: a transaction needs at least 1 output
display(query("""
SELECT output_count
FROM dataset
WHERE output_count < 1;
"""))

# wallet_age_days: a wallet can't be younger than 0 days
display(query("""
SELECT wallet_age_days
FROM dataset
WHERE wallet_age_days < 0;
"""))

# tx_count_24h: a count can't be negative
display(query("""
SELECT tx_count_24h
FROM dataset
WHERE tx_count_24h < 0;
"""))

# unique_counterparties_24h: a count can't be negative
display(query("""
SELECT unique_counterparties_24h
FROM dataset
WHERE unique_counterparties_24h < 0;
"""))

# avg_inter_tx_interval_s: time between transactions can't be negative
display(query("""
SELECT avg_inter_tx_interval_s
FROM dataset
WHERE avg_inter_tx_interval_s < 0;
"""))

# std_inter_tx_interval_s: a standard deviation can never be negative
display(query("""
SELECT std_inter_tx_interval_s
FROM dataset
WHERE std_inter_tx_interval_s < 0;
"""))

# balance_change: can be positive OR negative, so only check for missing values
display(query("""
SELECT balance_change
FROM dataset
WHERE balance_change IS NULL;
"""))

# new_address_fraction: a fraction must be between 0 and 1
display(query("""
SELECT new_address_fraction
FROM dataset
WHERE new_address_fraction < 0 OR new_address_fraction > 1;
"""))

# max_graph_hop: there must be at least 1 hop
display(query("""
SELECT max_graph_hop
FROM dataset
WHERE max_graph_hop < 1;
"""))

# time_since_prev_tx_s: time since the previous transaction can't be negative
display(query("""
SELECT time_since_prev_tx_s
FROM dataset
WHERE time_since_prev_tx_s < 0;
"""))

# gas_price_pctile: a percentile must be between 0 and 100
display(query("""
SELECT gas_price_pctile
FROM dataset
WHERE gas_price_pctile < 0 OR gas_price_pctile > 100;
"""))

# is_weekend: yes/no column, only 0 or 1 allowed
display(query("""
SELECT is_weekend
FROM dataset
WHERE is_weekend NOT IN (0, 1);
"""))

# kyc_tier: text column, so list every distinct value and how often it appears.

display(query("""
SELECT kyc_tier
FROM dataset
where kyc_tier not in ('basic', 'unverified', 'full')
"""))

# fraud: yes/no column, only 0 or 1 allowed
display(query("""
SELECT fraud
FROM dataset
WHERE fraud NOT IN (0, 1);
"""))



# from the reslut we can see that there are no inconsistencies because it doesnt give out the incosistencies we were trying to find

,amount


,tx_hour


,input_count


,output_count


,wallet_age_days


...

,time_since_prev_tx_s


,gas_price_pctile


,is_weekend


,kyc_tier


,fraud


In [ ]:
# we dont need to use imputation beause we don have NaN values, imputation repalces the nan columns with sth

In [ ]:
# checking almost all averages to see which columns have the biggest differences
display(query("""
select fraud,
       count(*)                               as n,
       avg(abs(amount))                       as avg_amount,
       avg(abs(tx_hour))                      as avg_tx_hour,
       avg(abs(input_count))                  as avg_input_count,
       avg(abs(output_count))                 as avg_output_count,
       avg(abs(wallet_age_days))              as avg_wallet_age_days,
       avg(abs(tx_count_24h))                 as avg_tx_count_24h,
       avg(abs(unique_counterparties_24h))    as avg_unique_counterparties_24h,
       avg(abs(avg_inter_tx_interval_s))      as avg_avg_inter_tx_interval_s,
       avg(abs(std_inter_tx_interval_s))      as avg_std_inter_tx_interval_s,
       avg(balance_change)                    as avg_balance_change,
       avg(abs(new_address_fraction))         as avg_new_address_fraction,
       avg(abs(max_graph_hop))                as avg_max_graph_hop,
       avg(abs(time_since_prev_tx_s))         as avg_time_since_prev_tx_s,
       avg(abs(gas_price_pctile))             as avg_gas_price_pctile
from dataset
group by fraud
order by fraud
"""))


,fraud,n,avg_amount,avg_tx_hour,avg_input_count,avg_output_count,avg_wallet_age_days,avg_tx_count_24h,avg_unique_counterparties_24h,avg_avg_inter_tx_interval_s,avg_std_inter_tx_interval_s,avg_balance_change,avg_new_address_fraction,avg_max_graph_hop,avg_time_since_prev_tx_s,avg_gas_price_pctile
0,0,225,6708.685244,12.208889,6.084444,7.573333,579.964444,13.240000,15.173333,3221.053333,2584.404444,-137.7472,0.275889,4.182222,891.488889,50.591111
1,1,175,19019.513257,12.348571,6.097143,7.405714,707.405714,11.011429,17.257143,2577.348571,3705.337143,2642.6416,0.390120,4.171429,1284.588571,48.805714


After analyzing the table we concluded that as the amount of the transaction is bigger ti is more likely to be fraudulent.

Also if the wallet_age_days is higher the more likely is that the transaction is fraudulent.

Moreover, the smaller the mean of the seconds between the wallets transactions is the smaller the bigger are the chances to be fraudulent

Furthermore, the standard devation is bigger the mroe likely is that the transaction is fraudulent. 

If the average balance change is negative it means that there are higher chances that the person is not a fraud while if they receive money (big amounts of money) it is often the case that this transaction is a fraud

if the proportion of the receiving addresses in a transaction that are brand new is big the higher the chances the transaction it is fraud

if the time period between the wallets transaction is small the higher the chances the transaction is fraudulent 

In [ ]:
# we are checkign whether there is a pattern with the account verification stage and the fraud column

display(query("""select count(fraud), kyc_tier, fraud
from dataset
group by kyc_tier, fraud
"""))

# The majority of fradulent transactions are the ones which the preson who made th transaction has verifcation state basic and also the unverified 
# accoutns are more likely to commit a fraud

,count(fraud),kyc_tier,fraud
0,104,basic,0
1,73,basic,1
2,76,full,0
3,46,full,1
4,45,unverified,0
5,56,unverified,1


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler 
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import mean_absolute_error as mae, mean_squared_error as mse
from math import sqrt

scaler = StandardScaler()
knn_reg = KNeighborsClassifier(n_neighbors=3)

def check_affect_one(df):
        # every non-numeric columns -> numeric
    numeric_columns = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
    non_numeric_columns = df.select_dtypes(exclude=['int64', 'float64']).columns.tolist()
    df_encoded = pd.get_dummies(df, columns=non_numeric_columns, dtype=int)

        # seperation on target vector and feature matrix
    df_y = df_encoded["fraud"] 
    df_X = df_encoded.drop(columns=["fraud"])
        
        # seperate test-train set by pattern 52
    df_X_train, df_X_test, df_y_train, df_y_test = train_test_split(df_X, df_y, train_size=0.8, random_state=52)

        # scaling values
    scaler.fit(df_X_train)
    df_X_train_scaled = scaler.transform(df_X_train)
    df_X_test_scaled = scaler.transform(df_X_test)

        # train model
    knn_reg.fit(df_X_train_scaled, df_y_train)

        # prediction
    df_y_pred = knn_reg.predict(df_X_test_scaled)

        # calculating MAE and RMSE
    mae_y = mae(df_y_test, df_y_pred)
    rmse_y = sqrt(mse(df_y_test, df_y_pred))

        # return the result
    return [mae_y, rmse_y]




def check_affect(orig, compar):
    origl = check_affect_one(orig)
    comparl = check_affect_one(compar)
    print(f"The first set's MAE: {origl[0]}, RMSE: {origl[1]}\nThe second set's MAE: {comparl[0]}, RMSE: {comparl[1]}")
    print(f"The difference between MAE: {origl[0] - comparl[0]}, and between RMSE: {origl[1] - comparl[1]}")

In [ ]:
columns = [
    "tx_hour",
    "input_count",
    "output_count",
    "tx_count_24h",
    "unique_counterparties_24h",
    "max_graph_hop",
    "gas_price_pctile",
    "is_weekend"
]

onlytheoensweneed = dataset.drop(columns,axis='columns')


In [ ]:
check_affect(dataset, onlytheoensweneed )
# based on what we analyzed this is our best chocie of columns that need to be dropped in order to have a better dataset

The first set's MAE: 0.35, RMSE: 0.5916079783099616
The second set's MAE: 0.3625, RMSE: 0.6020797289396148
The difference between MAE: -0.012500000000000011, and between RMSE: -0.01047175062965322


In [ ]:
columns = [
    "tx_hour",
    "input_count",
    "output_count",
    "tx_count_24h",
    "unique_counterparties_24h",
    "max_graph_hop",
    "gas_price_pctile",
    "is_weekend",
    "balance_change"
]

onlytheoensweneed = dataset.drop(columns,axis='columns')


In [ ]:
check_affect(dataset, onlytheoensweneed )
# the impact of the balance_change column is not very significant, but still it influences the result

The first set's MAE: 0.35, RMSE: 0.5916079783099616
The second set's MAE: 0.375, RMSE: 0.6123724356957945
The difference between MAE: -0.025000000000000022, and between RMSE: -0.02076445738583288


In [ ]:
columns = [
    "tx_hour",
    "input_count",
    "output_count",
    "tx_count_24h",
    "unique_counterparties_24h",
    "max_graph_hop",
    "gas_price_pctile",
    "is_weekend",
    "amount"
]

onlytheoensweneed = dataset.drop(columns,axis='columns')


In [ ]:
check_affect(dataset, onlytheoensweneed)
# the impact of amount column is not very significant, but still it influences the result

The first set's MAE: 0.35, RMSE: 0.5916079783099616
The second set's MAE: 0.375, RMSE: 0.6123724356957945
The difference between MAE: -0.025000000000000022, and between RMSE: -0.02076445738583288


In [ ]:
columns = [
    "tx_hour",
    "input_count",
    "output_count",
    "tx_count_24h",
    "unique_counterparties_24h",
    "max_graph_hop",
    "gas_price_pctile",
    "is_weekend",
    "std_inter_tx_interval_s"
]

onlytheoensweneed = dataset.drop(columns,axis='columns')


In [ ]:
check_affect(dataset, onlytheoensweneed)
# the impact of std_inter_tx_interval_s column is not very significant, but still it influences the result more than the previous columns

The first set's MAE: 0.35, RMSE: 0.5916079783099616
The second set's MAE: 0.3875, RMSE: 0.6224949798994366
The difference between MAE: -0.03750000000000003, and between RMSE: -0.030887001589474994


In [ ]:
columns = [
    "tx_hour",
    "input_count",
    "output_count",
    "tx_count_24h",
    "unique_counterparties_24h",
    "max_graph_hop",
    "gas_price_pctile",
    "is_weekend",
    "time_since_prev_tx_s"
]

onlytheoensweneed = dataset.drop(columns,axis='columns')

In [ ]:
check_affect(dataset, onlytheoensweneed)
# time_since_prev_tx_s as opposed to our assumptions that is useful for the prediction has a negative impact
# on the models prediction, it makes it deviate

The first set's MAE: 0.35, RMSE: 0.5916079783099616
The second set's MAE: 0.3375, RMSE: 0.5809475019311126
The difference between MAE: 0.012499999999999956, and between RMSE: 0.010660476378849038


In [ ]:
useful_columns = [
    "amount",
    "wallet_age_days",
    "avg_inter_tx_interval_s",
    "std_inter_tx_interval_s",
    "balance_change",
    "new_address_fraction",
    "kyc_tier"
]

# by calling vadim() we do backward elimination on the dataset which we think that until now has the best MAE, containing:
#wallet_age_days, avg_inter_tx_interval_s, std_inter_tx_interval_s, balance_change, new_address_fraction, kyc_tier, fraud

# vadim() only tells us what happens if we drop ONE column, but after it we never dropped anything.

def vadim():
    for column in useful_columns:
        print(column)
        
        onlytheoensweneed2 = onlytheoensweneed.drop(column,axis='columns')
        display(onlytheoensweneed2)
        
        check_affect(onlytheoensweneed, onlytheoensweneed2)

vadim()


# from the vadim we get that we need to drop the new_address_function as well from out best perfomign table

amount


,wallet_age_days,avg_inter_tx_interval_s,std_inter_tx_interval_s,balance_change,new_address_fraction,kyc_tier,fraud
0,23,32,44,-26124.58,0.358,full,0
1,505,1455,1252,-13905.04,0.405,unverified,1
2,170,285,502,1607.18,0.620,unverified,1
3,2578,957,1185,3996.99,0.500,basic,1
4,113,1060,931,-246.18,0.146,full,0
...,...,...,...,...,...,...,...
395,1420,4681,9568,-40321.86,0.226,full,1
396,55,7443,5602,14104.64,0.211,full,0
397,1979,7269,15471,-247.59,0.269,basic,1
398,35,4186,7809,-74462.25,0.242,unverified,1


The first set's MAE: 0.3375, RMSE: 0.5809475019311126
The second set's MAE: 0.325, RMSE: 0.570087712549569
The difference between MAE: 0.012500000000000011, and between RMSE: 0.010859789381543528
wallet_age_days


,amount,avg_inter_tx_interval_s,std_inter_tx_interval_s,balance_change,new_address_fraction,kyc_tier,fraud
0,46393.02,32,44,-26124.58,0.358,full,0
1,7485.22,1455,1252,-13905.04,0.405,unverified,1
2,872.26,285,502,1607.18,0.620,unverified,1
3,2763.08,957,1185,3996.99,0.500,basic,1
4,196.97,1060,931,-246.18,0.146,full,0
...,...,...,...,...,...,...,...
395,36041.28,4681,9568,-40321.86,0.226,full,1
396,8210.70,7443,5602,14104.64,0.211,full,0
397,162.45,7269,15471,-247.59,0.269,basic,1
398,42500.27,4186,7809,-74462.25,0.242,unverified,1


The first set's MAE: 0.3375, RMSE: 0.5809475019311126
The second set's MAE: 0.3625, RMSE: 0.6020797289396148
The difference between MAE: -0.024999999999999967, and between RMSE: -0.021132227008502258
avg_inter_tx_interval_s


...

The first set's MAE: 0.3375, RMSE: 0.5809475019311126
The second set's MAE: 0.3375, RMSE: 0.5809475019311126
The difference between MAE: 0.0, and between RMSE: 0.0
new_address_fraction


,amount,wallet_age_days,avg_inter_tx_interval_s,std_inter_tx_interval_s,balance_change,kyc_tier,fraud
0,46393.02,23,32,44,-26124.58,full,0
1,7485.22,505,1455,1252,-13905.04,unverified,1
2,872.26,170,285,502,1607.18,unverified,1
3,2763.08,2578,957,1185,3996.99,basic,1
4,196.97,113,1060,931,-246.18,full,0
...,...,...,...,...,...,...,...
395,36041.28,1420,4681,9568,-40321.86,full,1
396,8210.70,55,7443,5602,14104.64,full,0
397,162.45,1979,7269,15471,-247.59,basic,1
398,42500.27,35,4186,7809,-74462.25,unverified,1


The first set's MAE: 0.3375, RMSE: 0.5809475019311126
The second set's MAE: 0.3, RMSE: 0.5477225575051661
The difference between MAE: 0.03750000000000003, and between RMSE: 0.03322494442594648
kyc_tier


,amount,wallet_age_days,avg_inter_tx_interval_s,std_inter_tx_interval_s,balance_change,new_address_fraction,fraud
0,46393.02,23,32,44,-26124.58,0.358,0
1,7485.22,505,1455,1252,-13905.04,0.405,1
2,872.26,170,285,502,1607.18,0.620,1
3,2763.08,2578,957,1185,3996.99,0.500,1
4,196.97,113,1060,931,-246.18,0.146,0
...,...,...,...,...,...,...,...
395,36041.28,1420,4681,9568,-40321.86,0.226,1
396,8210.70,55,7443,5602,14104.64,0.211,0
397,162.45,1979,7269,15471,-247.59,0.269,1
398,42500.27,35,4186,7809,-74462.25,0.242,1


The first set's MAE: 0.3375, RMSE: 0.5809475019311126
The second set's MAE: 0.3625, RMSE: 0.6020797289396148
The difference between MAE: -0.024999999999999967, and between RMSE: -0.021132227008502258


In [ ]:
all_columns  = [
    "amount",
    "wallet_age_days",
    "avg_inter_tx_interval_s",
    "std_inter_tx_interval_s",
    "balance_change",
    "new_address_fraction",
    "kyc_tier",
    "tx_hour",
    "input_count",
    "output_count",
    "tx_count_24h",
    "unique_counterparties_24h",
    "max_graph_hop",
    "gas_price_pctile",
    "is_weekend",
    "time_since_prev_tx_s"
]


# by calling vadimGeneral() we do backward elimination on the whole dataset, to do an aditional check

# vadimGeneral() only tells us what happens if we drop ONE column, but after it we never dropped anything.
    
def vadimGeneral():
    for column in all_columns :
        print(column)
        
        onlytheoensweneed2 = dataset.drop(column,axis='columns')
        # display(onlytheoensweneed2)
        
        check_affect(dataset, onlytheoensweneed2)

vadimGeneral()


# starting from our 7 columns (which we picked with the sql queries) gives a better result

amount
The first set's MAE: 0.35, RMSE: 0.5916079783099616
The second set's MAE: 0.3625, RMSE: 0.6020797289396148
The difference between MAE: -0.012500000000000011, and between RMSE: -0.01047175062965322
wallet_age_days
The first set's MAE: 0.35, RMSE: 0.5916079783099616
The second set's MAE: 0.375, RMSE: 0.6123724356957945
The difference between MAE: -0.025000000000000022, and between RMSE: -0.02076445738583288
avg_inter_tx_interval_s
The first set's MAE: 0.35, RMSE: 0.5916079783099616
The second set's MAE: 0.375, RMSE: 0.6123724356957945
The difference between MAE: -0.025000000000000022, and between RMSE: -0.02076445738583288
std_inter_tx_interval_s
The first set's MAE: 0.35, RMSE: 0.5916079783099616
The second set's MAE: 0.4, RMSE: 0.6324555320336759
The difference between MAE: -0.050000000000000044, and between RMSE: -0.04084755372371429
balance_change
The first set's MAE: 0.35, RMSE: 0.5916079783099616
The second set's MAE: 0.3625, RMSE: 0.6020797289396148
The difference between M

We are starting the process of training the model

In [ ]:
##### transformation 

one_hot_encoding_copy = onlytheoensweneed.copy()

categorical_cols = [
   "kyc_tier"
]

# generate the dummy variables
one_hot_encoding_new = pd.get_dummies(one_hot_encoding_copy[categorical_cols],dtype=int)

# join old and new columns in one table
one_hot_encoding_new_concated = pd.concat([one_hot_encoding_copy, one_hot_encoding_new], axis=1)
# we delete the old kyc_toer string column
one_hot_encoding_new_concated = one_hot_encoding_new_concated.drop("kyc_tier", axis =1)
one_hot_encoding_new_concated




raw_data = one_hot_encoding_new_concated.copy()

In [ ]:
# last test to see if balance is useful or not in this dataframe: [["amount", "wallet_age_days", "avg_inter_tx_interval_s", "std_inter_tx_interval_s",
                             # "balance_chnage","kyc_tier", "fraud"]]


In [ ]:
final_dataset_col = dataset[["amount", "wallet_age_days", "avg_inter_tx_interval_s", "std_inter_tx_interval_s", "balance_change","kyc_tier", "fraud"]]

check_affect_one(final_dataset_col)

[0.3, 0.5477225575051661]

In [ ]:
# now without the balance
final_dataset_col = dataset[["amount", "wallet_age_days", "avg_inter_tx_interval_s", "std_inter_tx_interval_s",
                              "kyc_tier", "fraud"]]

check_affect_one(final_dataset_col)

[0.3, 0.5477225575051661]

In [ ]:
# so we found that the balance_change doesnt make a difference to the result so we are dropping it as well


# training the model
final_dataset_col = dataset[["amount", "wallet_age_days", "avg_inter_tx_interval_s", "std_inter_tx_interval_s",
                              "kyc_tier", "fraud"]]

check_affect_one(final_dataset_col)

[0.3, 0.5477225575051661]

we are saving the model and the scaler

In [ ]:
from joblib import dump, load

# Save the trained KNN model and the fitted scaler
dump(knn_reg, "knn_model.joblib")
dump(scaler, "scaler.joblib")

print("Model and scaler saved successfully!")

Model and scaler saved successfully!
